# Task 2 — Global vs Local Alignments
Compare Biopython pairwise alignments on Lab 1 sequences and capture interpretable fragments.

## 1. Initialize Project Environment
Import Biopython’s pairwise utilities, configure logging, and verify inputs.

In [13]:
from __future__ import annotations

import logging
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import pandas as pd
from Bio import SeqIO, pairwise2
from Bio.pairwise2 import format_alignment
from Bio.SeqRecord import SeqRecord

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")


def locate_data_root() -> Path:
    here = Path().resolve()
    for base in [here, *here.parents]:
        candidate = base / "data/work/AndreiCod/lab01"
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate data/work/AndreiCod/lab01 relative to current directory"
    )


DATA_ROOT = locate_data_root()
FASTAS = list(DATA_ROOT.glob("*.fa*"))
for f in FASTAS:
    logging.info("Found FASTA %s (%.2f MB)", f, f.stat().st_size / 1e6)

assert FASTAS, "No Lab 1 FASTA files located."


[INFO] Found FASTA /home/rbals/git/daha-bdhb/BDHB-lab/data/work/AndreiCod/lab01/nm000546.fa (0.00 MB)
[INFO] Found FASTA /home/rbals/git/daha-bdhb/BDHB-lab/data/work/AndreiCod/lab01/tp53_dna_multi.fasta (0.01 MB)
[INFO] Found FASTA /home/rbals/git/daha-bdhb/BDHB-lab/data/work/AndreiCod/lab01/tp53_protein_multi.fasta (0.00 MB)
[INFO] Found FASTA /home/rbals/git/daha-bdhb/BDHB-lab/data/work/AndreiCod/lab01/tp53_mm_protein.fasta (0.00 MB)
[INFO] Found FASTA /home/rbals/git/daha-bdhb/BDHB-lab/data/work/AndreiCod/lab01/tp53_hs_protein_P04637.fasta (0.00 MB)
[INFO] Found FASTA /home/rbals/git/daha-bdhb/BDHB-lab/data/work/AndreiCod/lab01/tp53_dr_protein.fasta (0.00 MB)
[INFO] Found FASTA /home/rbals/git/daha-bdhb/BDHB-lab/data/work/AndreiCod/lab01/tp53_hs_transcript_NM_000546.6.fasta (0.00 MB)
[INFO] Found FASTA /home/rbals/git/daha-bdhb/BDHB-lab/data/work/AndreiCod/lab01/tp53_dr_transcript_NM_131327.2.fasta (0.00 MB)
[INFO] Found FASTA /home/rbals/git/daha-bdhb/BDHB-lab/data/work/AndreiCod/l

## 2. Define Configuration Parameters
Declare which sequences to align, scoring settings, and output paths so reruns stay deterministic.

In [14]:
@dataclass
class AlignmentConfig:
    fasta_path: Path
    seq_indices: Tuple[int, int]
    match_score: int = 2
    mismatch_penalty: int = -1
    gap_open: int = -2
    gap_extend: float = -0.5
    keep_top: int = 1
    trim_to: Optional[int] = 3000  # limit sequence length to avoid exploding memory
    window_start: int = 0  # allow skipping large low-complexity prefixes

    def describe(self) -> Dict[str, str]:
        info = asdict(self)
        info["fasta_path"] = str(info["fasta_path"])
        return info


CONFIG = AlignmentConfig(
    fasta_path=DATA_ROOT / "my_tp53.fa",
    seq_indices=(0, 1),
)
CONFIG.describe()


{'fasta_path': '/home/rbals/git/daha-bdhb/BDHB-lab/data/work/AndreiCod/lab01/my_tp53.fa',
 'seq_indices': (0, 1),
 'match_score': 2,
 'mismatch_penalty': -1,
 'gap_open': -2,
 'gap_extend': -0.5,
 'keep_top': 1,
 'trim_to': 3000,
 'window_start': 0}

In [15]:
def slice_record(
    record: SeqRecord, start: int = 0, trim_to: Optional[int] = None
) -> SeqRecord:
    """Return a trimmed copy of the record without modifying the original."""
    if start < 0:
        raise ValueError("window_start must be >= 0")
    end = None if trim_to is None else start + trim_to
    if start == 0 and trim_to is None:
        return record
    sliced = record[start:end]
    sliced.description = f"{record.description} [slice {start}:{end or 'end'}]"
    return sliced


In [16]:
def load_sequences(cfg: AlignmentConfig):
    needed = set(cfg.seq_indices)
    rec_map: Dict[int, SeqRecord] = {}
    for idx, record in enumerate(SeqIO.parse(cfg.fasta_path, "fasta")):
        if idx in needed:
            rec_map[idx] = record
        if len(rec_map) == len(needed):
            break

    if len(rec_map) < len(needed):
        raise IndexError("Not enough sequences in FASTA for selected indices")

    i1, i2 = cfg.seq_indices
    full1, full2 = rec_map[i1], rec_map[i2]
    rec1 = slice_record(full1, cfg.window_start, cfg.trim_to)
    rec2 = slice_record(full2, cfg.window_start, cfg.trim_to)
    logging.info(
        "Selected %s (%d -> %d bp) and %s (%d -> %d bp)",
        full1.id,
        len(full1.seq),
        len(rec1.seq),
        full2.id,
        len(full2.seq),
        len(rec2.seq),
    )
    return rec1, rec2


seq_a, seq_b = load_sequences(CONFIG)
len(seq_a.seq), len(seq_b.seq)


[INFO] Selected NM_000546.6 (2512 -> 2512 bp) and NM_011640.3 (1781 -> 1781 bp)


(2512, 1781)

## 3. Implement Core Functionality
Run global and local alignments, capture scores, and highlight representative fragments for the report.

In [17]:
def run_alignments(seq1, seq2, cfg: AlignmentConfig):
    global_alignments = pairwise2.align.globalms(
        seq1.seq,
        seq2.seq,
        cfg.match_score,
        cfg.mismatch_penalty,
        cfg.gap_open,
        cfg.gap_extend,
        one_alignment_only=True,
    )[: cfg.keep_top]

    local_alignments = pairwise2.align.localms(
        seq1.seq,
        seq2.seq,
        cfg.match_score,
        cfg.mismatch_penalty,
        cfg.gap_open,
        cfg.gap_extend,
        one_alignment_only=True,
    )[: cfg.keep_top]

    return global_alignments, local_alignments


GLOBAL, LOCAL = run_alignments(seq_a, seq_b, CONFIG)
len(GLOBAL), len(LOCAL)


(1, 1)

In [18]:
def summarize_alignment(alignment) -> Dict[str, float]:
    seq1, seq2, score, start, end = alignment
    gaps_1 = seq1.count("-")
    gaps_2 = seq2.count("-")
    matches = sum(
        ch1 == ch2 for ch1, ch2 in zip(seq1, seq2) if ch1 != "-" and ch2 != "-"
    )
    return {
        "score": score,
        "start": start,
        "end": end,
        "length": len(seq1),
        "gaps_seq1": gaps_1,
        "gaps_seq2": gaps_2,
        "matches": matches,
    }


import pandas as pd

global_summary = pd.DataFrame([summarize_alignment(aln) for aln in GLOBAL])
local_summary = pd.DataFrame([summarize_alignment(aln) for aln in LOCAL])

global_summary

,score,start,end,length,gaps_seq1,gaps_seq2,matches
0,2010.0,0,2580,2580,68,799,1467


In [19]:
local_summary

,score,start,end,length,gaps_seq1,gaps_seq2,matches
0,2107.5,22,2069,2601,89,820,1404


In [20]:
def extract_alignment_fragment(
    global_aln, local_aln, window: int = 60
) -> Dict[str, str]:
    g_seq1, g_seq2, *_ = global_aln
    l_seq1, l_seq2, *_ = local_aln

    # naive strategy: take the first contiguous non-gap block from the local alignment
    start_idx = next(
        (i for i, (a, b) in enumerate(zip(l_seq1, l_seq2)) if a != "-" and b != "-"), 0
    )
    end_idx = min(len(l_seq1), start_idx + window)

    fragment = {
        "local_seq1": l_seq1[start_idx:end_idx],
        "local_seq2": l_seq2[start_idx:end_idx],
        "global_seq1": g_seq1[start_idx:end_idx],
        "global_seq2": g_seq2[start_idx:end_idx],
    }
    return fragment


fragment = extract_alignment_fragment(GLOBAL[0], LOCAL[0], window=80)
fragment

{'local_seq1': 'CT--CAAAAGT-CTAG-AGCCACCGTCCA--GGGAGCA----G---GTAGCT-GCTGGGCTCC-----GGGGACACTTTG',
 'local_seq2': 'CTGGCTAAAGTTCT-GTAGCTTCAGTTCATTGGGACCATCCTGGCTGTAGGTAGC--GACTACAGTTAGGGGGCACCTAG',
 'global_seq1': '------AAAGT-CTAG-AGCCACCGTCCA--GGGAGCA----G---GTAGCT-GCTGGGCTCC-----GGGGACACTTTG',
 'global_seq2': 'CTGGCTAAAGTTCT-GTAGCTTCAGTTCATTGGGACCATCCTGGCTGTAGGTAGC--GACTACAGTTAGGGGGCACCTAG'}

In [21]:
print("=== Global Alignment (top) ===")
print(format_alignment(*GLOBAL[0]))
print("=== Local Alignment (top) ===")
print(format_alignment(*LOCAL[0]))

=== Global Alignment (top) ===
----------------CTCA--------AAAGT-CTAG-AGCCACCGTCCA--GGGAGCA----G---GTAGCT-GCTGGGCTCC-----GGGGACACTTTGCGTTCGGGCTGGGAGCGTGCT--TTCCAC-G-ACGGTGACACGCTTC-CCTGGATTGGCAGCCAGACTGCCTTCCGGGTCACTGCCATGGAGGAGCCGCAGTCAGATCCT-AGCGTCGAGCCCCCTCTGAGTCAGGAAACATTTTCAGACCTATGGAAACTACTTCCT---GAAAACAACGTTCTGTCCCCCTTGCCGTCCCAAGCAATGGATGATTTGATGCTGTCCCCGGACGATATTGAACAATGGTTCACTGAAGACCCAGGTCCAGATGAAGCTCC-CAGAATGCCAGAG-GCTGCTCCCCCCGTGGCCCCTG-CACCAGCAG-CTCCTACACCGGCGGCCCCTGCACCAGCCCCCTCC-TGGCCCCTGTCATCTTCTGTCCCTTCCCAGAAAACCTACCAGGGCAGCTACGGTTTCCGTCTGGGCTTCTTGCATTCTGGGACAGCCAAGTCTGTGACT-TGCACGTACTCCCCTGCCCTCAACAAGATGTTTTGCCAACTGGCCAAGACCTGCCCTGTGCAGCTGTGGGTTGATT-CCACACCCCCGCCCGGCACCCGCGTCCGCGCCATGGCCATCTACAAGCAGTCACAGCACATGACGGAGGTTGTGAGGCGCTGCCCCCACCATGAGCGCTGCTCAGATAGCGATGGTCTGGCCCCTCCTCAGCATCTTATCCGAGTGGAAGGAAATTTGCGTGTGGAGTATTTGGATGACAGAAACACTTTTCGACATAGTGTGGTGGTGCCCTATGAGCCGCCTGAGGTTGGCTCTGACTGTACCACCATCCACTACAACTACATGTGTAACAGTTCCTGCATGGGCGGCATGAACCGGAGGCCCATCCTCACCATCATCACACT

## 4. Validate with Unit Tests
Use quick checks to ensure alignment helpers behave deterministically and fragments cover actual matches.

In [22]:
def test_run_alignments_on_short_seq():
    records = list(SeqIO.parse(DATA_ROOT / "nm000546.fa", "fasta"))
    cfg = AlignmentConfig(fasta_path=DATA_ROOT / "nm000546.fa", seq_indices=(0, 0))
    g, l = run_alignments(records[0], records[0], cfg)
    assert g[0][2] >= l[0][2]


def test_fragment_contains_match():
    frag = extract_alignment_fragment(GLOBAL[0], LOCAL[0], window=40)
    assert frag["local_seq1"].replace("-", "")


test_run_alignments_on_short_seq()
test_fragment_contains_match()
print("Alignment helper tests passed.")

Alignment helper tests passed.


## 5. Analyze Performance Metrics
Benchmark global vs local execution time and create a quick comparison table for reporting.

In [23]:
import time


def timed(fn, *args, repeats: int = 3, **kwargs):
    durations = []
    for _ in range(repeats):
        start = time.perf_counter()
        fn(*args, **kwargs)
        durations.append(time.perf_counter() - start)
    return sum(durations) / len(durations)


mean_global = timed(
    lambda: pairwise2.align.globalms(
        seq_a.seq,
        seq_b.seq,
        CONFIG.match_score,
        CONFIG.mismatch_penalty,
        CONFIG.gap_open,
        CONFIG.gap_extend,
    )
)

mean_local = timed(
    lambda: pairwise2.align.localms(
        seq_a.seq,
        seq_b.seq,
        CONFIG.match_score,
        CONFIG.mismatch_penalty,
        CONFIG.gap_open,
        CONFIG.gap_extend,
    )
)

pd.DataFrame(
    [
        {"type": "global", "mean_runtime_s": mean_global},
        {"type": "local", "mean_runtime_s": mean_local},
    ]
)

,type,mean_runtime_s
0,global,0.412324
1,local,0.910466


In [24]:
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

with open(ARTIFACT_DIR / "task2_global_alignment.txt", "w") as fh:
    fh.write(format_alignment(*GLOBAL[0]))
with open(ARTIFACT_DIR / "task2_local_alignment.txt", "w") as fh:
    fh.write(format_alignment(*LOCAL[0]))

pd.DataFrame([fragment]).to_csv(ARTIFACT_DIR / "task2_fragment.csv", index=False)
print("Saved alignment artifacts to", ARTIFACT_DIR)

Saved alignment artifacts to artifacts
